## Change image format
After downloading images, some are different format to png. This code will convert all images in the folder to .png format

In [17]:
import os
import subprocess

# --- CONFIGURATION ---
SUB_FOLDER = "/Users/vik/Downloads/testfolder/player_portraits"
# ---------------------

def standardize_to_png():
    files = os.listdir(SUB_FOLDER)
    converted_count = 0

    # List of "annoying" modern formats
    target_extensions = ('.avif', '.webp', 'jpeg')

    print(f"Checking for modern formats in: {SUB_FOLDER}...")

    for filename in files:
        if filename.lower().endswith(target_extensions):
            input_path = os.path.join(SUB_FOLDER, filename)
            
            # Step 1: Fix the naming
            # This turns "image.png.avif" -> "image.png"
            # Or "photo.webp" -> "photo.png"
            new_filename = filename.rsplit('.', 1)[0]
            if not new_filename.lower().endswith('.png'):
                new_filename += ".png"
            
            output_path = os.path.join(SUB_FOLDER, new_filename)

            # Step 2: Use macOS 'sips' to convert
            try:
                # sips -s format [type] [input] --out [output]
                subprocess.run([
                    "sips", "-s", "format", "png", 
                    input_path, "--out", output_path
                ], check=True, capture_output=True)
                
                # Step 3: Delete the original modern file
                os.remove(input_path)
                converted_count += 1
                print(f"✅ Converted & Replaced: {filename} -> {new_filename}")
                
            except Exception as e:
                print(f"❌ Failed to convert {filename}: {e}")

    if converted_count == 0:
        print("No .avif or .webp files found to convert.")
    else:
        print(f"\nFinished! {converted_count} files are now standard PNGs.")

if __name__ == "__main__":
    standardize_to_png()

Checking for modern formats in: /Users/vik/Downloads/testfolder/player_portraits...
✅ Converted & Replaced: tombanton.webp -> tombanton.png
✅ Converted & Replaced: benstokes.webp -> benstokes.png
✅ Converted & Replaced: moeenali.webp -> moeenali.png
✅ Converted & Replaced: markwood.webp -> markwood.png
✅ Converted & Replaced: samcurran.webp -> samcurran.png
✅ Converted & Replaced: reecetopley.webp -> reecetopley.png
✅ Converted & Replaced: liamdawson.webp -> liamdawson.png
✅ Converted & Replaced: willjacks.webp -> willjacks.png
✅ Converted & Replaced: jonnybairstow.webp -> jonnybairstow.png
✅ Converted & Replaced: liamlivingstone.webp -> liamlivingstone.png
✅ Converted & Replaced: harrybrook.webp -> harrybrook.png
✅ Converted & Replaced: jofraarcher.webp -> jofraarcher.png
✅ Converted & Replaced: olliepope.webp -> olliepope.png
✅ Converted & Replaced: jamieoverton.webp -> jamieoverton.png
✅ Converted & Replaced: josbuttler.webp -> josbuttler.png
✅ Converted & Replaced: dawidmalan.webp 

## Put logo in the background of portraits
This could be used with original portraits and also with the cartoon styles. This generates a static logo background image. 
We further below have a version where we create a gif file as well with logo animating in the background of still cartoon of player.

In [18]:
import os
from PIL import Image

# --- CONFIGURATION ---
MAIN_FOLDER = "/Users/vik/Downloads/testfolder"
SUB_FOLDER = "/Users/vik/Downloads/testfolder/testimages"
BG_FILENAME = "BG.png"
OUTPUT_FOLDER = "/Users/vik/Downloads/testfolder/output"
TARGET_SIZE = (512, 512) 
# ---------------------

def process_images():
    # 1. Setup Folders
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)

    bg_path = os.path.join(MAIN_FOLDER, BG_FILENAME)
    
    if not os.path.exists(bg_path):
        print(f"❌ Error: Background image 'BG.png' not found in {MAIN_FOLDER}")
        return

    # 2. Open Background
    with Image.open(bg_path).convert("RGBA") as bg:
        bg_w, bg_h = bg.size

        # 3. Loop through subfolder
        found_files = 0
        for filename in os.listdir(SUB_FOLDER):
            # Process common image formats
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                found_files += 1
                subject_path = os.path.join(SUB_FOLDER, filename)
                
                try:
                    with Image.open(subject_path).convert("RGBA") as subject:
                        # A. Resize subject to match Background Height
                        subj_w, subj_h = subject.size
                        aspect_ratio = subj_w / subj_h
                        new_width = int(bg_h * aspect_ratio)
                        
                        subject_resized = subject.resize((new_width, bg_h), Image.Resampling.LANCZOS)
                        
                        # B. Composite onto Background
                        final_canvas = bg.copy()
                        offset_x = (bg_w - new_width) // 2
                        final_canvas.alpha_composite(subject_resized, (offset_x, 0))
                        
                        # C. Resize entire image to 512x512
                        final_resized = final_canvas.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
                        
                        # D. Save using the EXACT original filename
                        save_path = os.path.join(OUTPUT_FOLDER, filename)
                        
                        # Force PNG format if you want consistent transparency
                        # Or use final_resized.save(save_path) to keep original extension logic
                        final_resized.save(save_path, "PNG", optimize=True)
                        
                        print(f"✨ Processed and saved: {filename}")

                except Exception as e:
                    print(f"⚠️ Error processing {filename}: {e}")

        if found_files == 0:
            print(f"❓ No valid images found in {SUB_FOLDER}.")
        else:
            print(f"\n✅ All done! {found_files} images processed at 512x512.")

if __name__ == "__main__":
    process_images()

✨ Processed and saved: viratkohli.png
✨ Processed and saved: ravichandranashwin.png
✨ Processed and saved: hardikpandya.png
✨ Processed and saved: ishankishan.png
✨ Processed and saved: klrahul.png
✨ Processed and saved: arshdeepsingh.png
✨ Processed and saved: shreyasiyer.png
✨ Processed and saved: ravindrajadeja.png
✨ Processed and saved: sanjusamson.png
✨ Processed and saved: tilakvarma.png
✨ Processed and saved: shubmangill.png
✨ Processed and saved: varunchakravarthy.png
✨ Processed and saved: mohammedshami.png
✨ Processed and saved: axarpatel.png
✨ Processed and saved: jaspritbumrah.png
✨ Processed and saved: abhisheksharma.png
✨ Processed and saved: shubmangill.png.png
✨ Processed and saved: suryakumaryadav.png
✨ Processed and saved: shivamdube.png
✨ Processed and saved: prasidhkrishna.png
✨ Processed and saved: rohitsharma.png
✨ Processed and saved: jonnybairstow.png
✨ Processed and saved: mohammedsiraj.png
✨ Processed and saved: shardulthakur.png

✅ All done! 24 images process

## Cartoonize a single player face

In [25]:
import cv2
import numpy as np
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1)

def clean_cartoon_no_border(image_path, output_path):
    img = cv2.imread(image_path)
    if img is None: return
    h, w, _ = img.shape
    
    # 1. Smooth colors aggressively
    smoothed = cv2.bilateralFilter(img, d=15, sigmaColor=150, sigmaSpace=150)
    
    # 2. Quantize colors (K-Means)
    data = np.float32(smoothed).reshape((-1, 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 0.001)
    ret, label, center = cv2.kmeans(data, 10, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    center = np.uint8(center)
    cartoon_base = center[label.flatten()].reshape((smoothed.shape))

    # 3. Add Feature Lines ONLY
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_img)
    
    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark
        
        def get_pts(indices):
            return np.array([[int(landmarks[i].x * w), int(landmarks[i].y * h)] for i in indices], np.int32)

    cv2.imwrite(output_path, cartoon_base)
    print(f"Lineless Cartoon saved: {output_path}")

clean_cartoon_no_border('/Users/vik/Downloads/testfolder/player_portraits/adilrashid.png', 
              '/Users/vik/Downloads/testfolder/player_cartoons/adilrashid.png')

I0000 00:00:1777558393.505909       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Max


Lineless Cartoon saved: /Users/vik/Downloads/testfolder/player_cartoons/adilrashid.png


## Render frames for a logo
This is an optional command - where we could get render frames of logo animation. Could be used for team name displays as gif.

In [18]:
import cv2
import numpy as np
import os

def animate_logo_optimized(logo_path, output_folder):
    # 1. Load the transparent logo
    img = cv2.imread(logo_path, cv2.IMREAD_UNCHANGED)
    if img is None or img.shape[2] != 4:
        print("Please use a transparent PNG logo.")
        return

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    h, w = img.shape[:2]
    
    # 2. Generate 30 frames
    for frame in range(30):
        padding = 50
        canvas = np.zeros((h + padding*2, w + padding*2, 4), dtype=np.uint8)
        
        # Sine wave math for effects
        pulse = (np.sin(frame * (2 * np.pi / 30)) + 1) / 2
        
        # 3. Create the Glow (Vectorized)
        b, g, r, a = cv2.split(img)
        glow_mask = cv2.GaussianBlur(a, (51, 51), 0)
        glow_alpha = (glow_mask * pulse * 0.6).astype(np.uint8)
        
        # Build the glow layer
        y_off, x_off = padding, padding
        glow_color = [255, 180, 50] # Gold/Orange glow
        for i in range(3):
            canvas[y_off:y_off+h, x_off:x_off+w, i] = glow_color[i]
        canvas[y_off:y_off+h, x_off:x_off+w, 3] = glow_alpha

        # 4. Apply Wave Displacement (Vectorized - No more IndexErrors!)
        # Create coordinate grids
        grid_x, grid_y = np.meshgrid(np.arange(w), np.arange(h))
        
        # Apply the sine wave math to the entire grid at once
        # This shifts the X coordinates based on the Y position
        map_x = grid_x.astype(np.float32) + np.sin(grid_y / 30.0 + frame / 5.0).astype(np.float32) * 8
        map_y = grid_y.astype(np.float32)
        
        # Remap the pixels
        distorted_logo = cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR)
        
        # 5. Fast Alpha Blending
        logo_a = distorted_logo[:, :, 3] / 255.0
        for c in range(3):
            canvas[y_off:y_off+h, x_off:x_off+w, c] = (
                distorted_logo[:, :, c] * logo_a + 
                canvas[y_off:y_off+h, x_off:x_off+w, c] * (1 - logo_a)
            ).astype(np.uint8)
        
        canvas[y_off:y_off+h, x_off:x_off+w, 3] = np.maximum(
            canvas[y_off:y_off+h, x_off:x_off+w, 3], 
            distorted_logo[:, :, 3]
        )

        cv2.imwrite(f"{output_folder}/frame_{frame:02d}.png", canvas)

    print(f"✅ Animation complete! Frames saved to: {output_folder}")

# Run it
animate_logo_optimized('/Users/vik/Downloads/testfolder/BG.png', '/Users/vik/Downloads/testfolder/logorender')

✅ Animation complete! Frames saved to: /Users/vik/Downloads/testfolder/logorender


## Render player anime onto logo
This generates individual frames for player face infront of animated logo


In [30]:
import cv2
import numpy as np
import os

def animated_logo_generic(player_path, logo_path, output_folder):
    # 1. Load images
    player = cv2.imread(player_path, cv2.IMREAD_UNCHANGED)
    logo = cv2.imread(logo_path, cv2.IMREAD_UNCHANGED)
    
    if player is None or logo is None:
        print("Error: Could not find images. Check your paths.")
        return

    # 2. DYNAMICALLY get logo dimensions
    logo_h, logo_w = logo.shape[:2]
    
    # Define padding as a percentage of the logo height (e.g., 5%) 
    # This keeps padding proportional even if the logo is massive
    pad_val = int(logo_h * 0.05)
    canvas_h, canvas_w = logo_h + (pad_val * 2), logo_w + (pad_val * 2)

    # 3. DYNAMICALLY scale player to match logo height
    target_h = logo_h 
    player_orig_h, player_orig_w = player.shape[:2]
    aspect_ratio = player_orig_w / player_orig_h
    target_w = int(target_h * aspect_ratio)
    
    # Resize player using high-quality interpolation
    player_resized = cv2.resize(player, (target_w, target_h), interpolation=cv2.INTER_LANCZOS4)
    ph, pw = player_resized.shape[:2]

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Pre-split logo channels for the premium movement effect
    b, g, r, a = cv2.split(logo)

    for frame in range(30):
        # Create canvas based on dynamic dimensions
        canvas = np.zeros((canvas_h, canvas_w, 4), dtype=np.uint8)
        
        # --- PART A: LOGO MOTION & GLOW ---
        pulse = (np.sin(frame * (2 * np.pi / 30)) + 1) / 2
        # Blur kernel size scales with logo size for consistency
        blur_size = int(logo_h * 0.03) | 1 # Must be odd
        glow_mask = cv2.GaussianBlur(a, (blur_size, blur_size), 0)
        glow_alpha = (glow_mask * pulse * 0.6).astype(np.uint8)
        
        ly, lx = pad_val, pad_val
        glow_color = [255, 180, 50] 
        for i in range(3):
            canvas[ly:ly+logo_h, lx:lx+logo_w, i] = glow_color[i]
        canvas[ly:ly+logo_h, lx:lx+logo_w, 3] = glow_alpha

        # Wave Displacement
        grid_x, grid_y = np.meshgrid(np.arange(logo_w), np.arange(logo_h))
        # Wave strength scales with logo width
        wave_strength = logo_w * 0.008 
        map_x = grid_x.astype(np.float32) + np.sin(grid_y / (logo_h/40) + frame / 5.0).astype(np.float32) * wave_strength
        map_y = grid_y.astype(np.float32)
        distorted_logo = cv2.remap(logo, map_x, map_y, cv2.INTER_LINEAR)
        
        # Blend Logo
        logo_a = distorted_logo[:, :, 3] / 255.0
        logo_roi = canvas[ly:ly+logo_h, lx:lx+logo_w]
        for c in range(3):
            logo_roi[:, :, c] = (distorted_logo[:, :, c] * logo_a + 
                                logo_roi[:, :, c] * (1 - logo_a)).astype(np.uint8)
        logo_roi[:, :, 3] = np.maximum(logo_roi[:, :, 3], distorted_logo[:, :, 3])

        # --- PART B: PLAYER OVERLAY ---
        # Centering player width-wise on the canvas
        py = pad_val
        px = (canvas_w - pw) // 2
        
        player_bgr = player_resized[:, :, :3]
        player_a = player_resized[:, :, 3] / 255.0
        
        player_roi = canvas[py:py+ph, px:px+pw]
        for c in range(3):
            player_roi[:, :, c] = (player_bgr[:, :, c] * player_a + 
                                  player_roi[:, :, c] * (1 - player_a)).astype(np.uint8)
        player_roi[:, :, 3] = np.maximum(player_roi[:, :, 3], player_resized[:, :, 3])

        cv2.imwrite(f"{output_folder}/frame_{frame:02d}.png", canvas)

    print(f"✅ Rendered at {canvas_w}x{canvas_h}. Player automatically scaled to match {logo_h}px logo.")

# Run
animated_logo_generic('/Users/vik/Downloads/testfolder/output/chrisjordan_clean.png',
    '/Users/vik/Downloads/testfolder/England.png',
    '/Users/vik/Downloads/testfolder/full_height_render/')

ValueError: operands could not be broadcast together with shapes (2000,436) (2000,2083) 

In [39]:
import cv2
import numpy as np
import os

def animated_full_bleed_background(player_path, logo_path, output_folder, target_height=1024):
    # 1. Load images
    player = cv2.imread(player_path, cv2.IMREAD_UNCHANGED)
    logo = cv2.imread(logo_path, cv2.IMREAD_UNCHANGED)
    
    if player is None or logo is None:
        print("Error: Could not find images.")
        return

    # 2. Standardize Player (The Anchor)
    p_orig_h, p_orig_w = player.shape[:2]
    p_aspect = p_orig_w / p_orig_h
    ph = target_height
    pw = int(ph * p_aspect)
    player_resized = cv2.resize(player, (pw, ph), interpolation=cv2.INTER_LANCZOS4)

    # 3. Canvas Setup
    pad_val = int(target_height * 0.0) # change value here for padding if needed
    canvas_w = pw + (pad_val * 2)
    canvas_h = ph + (pad_val * 2)

    # 4. Logo Freedom (Scale and Crop to Fill Canvas)
    l_h, l_w = logo.shape[:2]
    # Calculate scale to cover the canvas (Fill/Cover logic)
    scale = max(canvas_w / l_w, canvas_h / l_h)
    new_lw, new_lh = int(l_w * scale), int(l_h * scale)
    logo_scaled = cv2.resize(logo, (new_lw, new_lh), interpolation=cv2.INTER_AREA)
    
    # Center Crop the logo to match canvas exactly
    start_x = (new_lw - canvas_w) // 2
    start_y = (new_lh - canvas_h) // 2
    logo_cropped = logo_scaled[start_y:start_y+canvas_h, start_x:start_x+canvas_w]
    
    lh, lw = logo_cropped.shape[:2]

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Split channels for the glow/motion
    b, g, r, a = cv2.split(logo_cropped)

    for frame in range(30):
        canvas = np.zeros((canvas_h, canvas_w, 4), dtype=np.uint8)
        
        # --- BACKGROUND LOGO MOTION ---
        pulse = (np.sin(frame * (2 * np.pi / 30)) + 1) / 2
        blur_size = int(canvas_h * 0.03) | 1
        # glow_mask = cv2.GaussianBlur(a, (blur_size, blur_size), 0)
        # glow_alpha = (glow_mask * pulse * 0.5).astype(np.uint8)
        
        # Glow (Full Canvas)
        # glow_color = [255, 180, 50] 
        # for i in range(3):
        #     canvas[:, :, i] = glow_color[i]
        # canvas[:, :, 3] = glow_alpha

        # Wave Displacement (Applied to the full cropped background)
        grid_x, grid_y = np.meshgrid(np.arange(canvas_w), np.arange(canvas_h))
        map_x = grid_x.astype(np.float32) + np.sin(grid_y / 40.0 + frame / 5.0).astype(np.float32) * (canvas_w * 0.01)
        map_y = grid_y.astype(np.float32)
        distorted_bg = cv2.remap(logo_cropped, map_x, map_y, cv2.INTER_LINEAR)
        
        # Blend Background
        bg_a = distorted_bg[:, :, 3] / 255.0
        for c in range(3):
            canvas[:, :, c] = (distorted_bg[:, :, c] * bg_a + 
                               canvas[:, :, c] * (1 - bg_a)).astype(np.uint8)
        canvas[:, :, 3] = np.maximum(canvas[:, :, 3], distorted_bg[:, :, 3])

        # --- PLAYER OVERLAY ---
        py, px = pad_val, pad_val
        player_bgr = player_resized[:, :, :3]
        player_a = player_resized[:, :, 3] / 255.0
        
        player_roi = canvas[py:py+ph, px:px+pw]
        for c in range(3):
            player_roi[:, :, c] = (player_bgr[:, :, c] * player_a + 
                                  player_roi[:, :, c] * (1 - player_a)).astype(np.uint8)
        player_roi[:, :, 3] = np.maximum(player_roi[:, :, 3], player_resized[:, :, 3])

        cv2.imwrite(f"{output_folder}/full_bleed_{frame:02d}.png", canvas)

    print(f"✅ Render Complete. Logo now fills the {canvas_w}x{canvas_h} area.")

# Run
animated_full_bleed_background('/Users/vik/Downloads/testfolder/player_cartoons/harrybrook.png',
    '/Users/vik/Downloads/testfolder/England.png',
    '/Users/vik/Downloads/testfolder/full_height_render/')

✅ Render Complete. Logo now fills the 1024x1024 area.


## Generating a gif

It used rendered frames from a given folder to give an output gif file.

In [40]:
from PIL import Image
import glob
import os

def create_test_gif(frame_folder, output_gif_path):
    # 1. Grab all frames in order
    frame_files = sorted(glob.glob(os.path.join(frame_folder, "full_bleed_*.png")))
    
    if not frame_files:
        print("No frames found! Check your folder path.")
        return

    # 2. Open images and handle transparency
    frames = []
    for f in frame_files:
        img = Image.open(f)
        # Ensure we keep the transparent background
        frames.append(img)

    # 3. Save as a looping GIF
    # duration=40 is approx 25 frames per second
    # loop=0 means loop forever
    # disposal=2 is CRITICAL for transparent GIFs to prevent "ghosting"
    frames[0].save(
        output_gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=40,
        loop=0,
        disposal=2 
    )
    print(f"✅ Test GIF created: {output_gif_path}")

# Run the test
create_test_gif('/Users/vik/Downloads/testfolder/full_height_render/', '/Users/vik/Downloads/testfolder/full_height_render.gif')

✅ Test GIF created: /Users/vik/Downloads/testfolder/full_height_render.gif


# Batch Run procedure
From player portraits to player cartoons


In [1]:
import cv2
import numpy as np
import glob
import os

def process_cartoon(img, spatial_radius=20, color_radius=30, num_colors=8):
    """
    Applies the specific cartoon filter logic from the UI app.
    """
    # 1. Background Masking (Matches UI: treats very white areas as background)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 250, 255, cv2.THRESH_BINARY)
    
    # 2. Mean Shift Filtering (The core smoothing step)
    # This is more computationally expensive than bilateral but gives that 'painted' look
    shifted = cv2.pyrMeanShiftFiltering(img, sp=spatial_radius, sr=color_radius)
    
    # 3. K-Means Quantization
    data = np.float32(shifted).reshape((-1, 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, label, center = cv2.kmeans(data, num_colors, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    
    # Reconstruct image
    res = np.uint8(center)[label.flatten()].reshape((shifted.shape))
    
    # 4. Re-apply background mask (keeps backgrounds pure white if they were originally)
    res[mask == 255] = [255, 255, 255]
    
    return res

def batch_process(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created directory: {output_folder}")

    files = glob.glob(os.path.join(input_folder, "*.png"))
    
    if not files:
        print("No PNG files found.")
        return

    # Use the defaults from your Streamlit app
    SP = 20
    SR = 30
    K = 8

    for image_path in files:
        img = cv2.imread(image_path)
        if img is None: 
            continue
        
        # Apply the UI-identical logic
        result = process_cartoon(img, spatial_radius=SP, color_radius=SR, num_colors=K)
        
        filename = os.path.basename(image_path)
        save_path = os.path.join(output_folder, filename)
        
        cv2.imwrite(save_path, result)
        print(f"Processed: {filename}")

# --- Configuration ---
input_dir = '/Users/vik/Downloads/testfolder/player_portraits/england'
output_dir = '/Users/vik/Downloads/testfolder/player_cartoons'


if __name__ == "__main__":
    batch_process(input_dir, output_dir)

Processed: ollierobinson.png
Processed: lukewood.png
Processed: shoaibbashir.png
Processed: joeroot.png
Processed: jofraarcher.png
Processed: chriswoakes.png
Processed: brydoncarse.png
Processed: markwood.png
Processed: adilrashid.png
Processed: rehanahmed.png
Processed: gusatkinson.png
Processed: samcurran.png
Processed: dawidmalan.png
Processed: philsalt.png
Processed: josbuttler.png
Processed: joshtongue.png
Processed: liamlivingstone.png
Processed: moeenali.png
Processed: benstokes.png
Processed: willjacks.png
Processed: chrisjordan.png
Processed: jamieoverton.png
Processed: jacobbethell.png
Processed: harrybrook.png
Processed: jonnybairstow.png
Processed: olliepope.png
Processed: liamdawson.png
Processed: benduckett.png
Processed: tombanton.png


# Batch GIF procedure
From Player cartoons to GIFs

In [6]:
import cv2
import numpy as np
import os
import glob
from PIL import Image

def batch_process_to_gif(input_folder, logo_path, output_folder, target_height=1024, target_width=1024):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    logo_img = cv2.imread(logo_path, cv2.IMREAD_UNCHANGED)
    if logo_img is None:
        print(f"❌ Error: Could not read logo at {logo_path}")
        return
    print(f"🖼️ Logo Loaded: {logo_path} | Shape: {logo_img.shape}")

    player_paths = glob.glob(os.path.join(input_folder, "*.png"))
    print(f"🔍 Found {len(player_paths)} files. Starting processing...")

    for p_path in player_paths:
        file_name = os.path.splitext(os.path.basename(p_path))[0]
        player_img = cv2.imread(p_path, cv2.IMREAD_UNCHANGED)
        
        if player_img is None:
            print(f"❌ {file_name}: OpenCV could not read file.")
            continue

        # --- DEBUG PRINT ---
        p_h, p_w = player_img.shape[:2]
        channels = player_img.shape[2] if len(player_img.shape) > 2 else 1
        print(f"Processing: {file_name} | Orig: {p_w}x{p_h} | Channels: {channels}")
        
        # 1. Player Resize
        scale = target_height / p_h
        new_pw = int(p_w * scale)
        player_resized_temp = cv2.resize(player_img, (new_pw, target_height), interpolation=cv2.INTER_LANCZOS4)

        # 2. Logo Logic
        l_h, l_w = logo_img.shape[:2]
        logo_scale = target_height / l_h
        rescaled_logo_w = int(l_w * logo_scale)
        logo_rescaled = cv2.resize(logo_img, (rescaled_logo_w, target_height), interpolation=cv2.INTER_AREA)
        
        if rescaled_logo_w >= target_width:
            start_x = (rescaled_logo_w - target_width) // 2
            logo_final = logo_rescaled[0:target_height, start_x:start_x + target_width]
        else:
            logo_final = np.zeros((target_height, target_width, 4), dtype=np.uint8)
            start_x = (target_width - rescaled_logo_w) // 2
            logo_final[0:target_height, start_x:start_x + rescaled_logo_w] = logo_rescaled

        # # 3. Center the Player (FIXED ASSIGNMENT)
        # player_resized = np.zeros((target_height, target_width, 4), dtype=np.uint8)
        # p_start_x = (target_width - new_pw) // 2
        
        # if new_pw > target_width:
        #      p_crop = player_resized_temp[0:target_height, (new_pw-target_width)//2 : (new_pw-target_width)//2 + target_width]
        #      player_resized = p_crop
        #      print(f"   ⚠️ {file_name} is wider than canvas. Cropped.")
        # else:
        #      # FIXED: Added the '=' assignment here
        #      player_resized[0:target_height, p_start_x:p_start_x + new_pw] = player_resized_temp
        #      print(f"   ✅ {file_name} centered at x={p_start_x}")
        # 3. Placement Logic
        player_resized = np.zeros((target_height, target_width, 4), dtype=np.uint8)
        
        # LOGIC: If you want to see the logo, don't put the player in the exact center
        # Let's try placing the player on the RIGHT THIRD of the canvas
        p_start_x = int(target_width * 0.6) - (new_pw // 2) 
        
        # Safety: Ensure the player doesn't go off-screen
        p_start_x = max(0, min(p_start_x, target_width - new_pw))

        print(f"   📏 Width Comparison: Logo Canvas={target_width}px | Player={new_pw}px")
        print(f"   📍 Placing player at x-coordinate: {p_start_x}")

        if new_pw > target_width:
             p_crop = player_resized_temp[0:target_height, (new_pw-target_width)//2 : (new_pw-target_width)//2 + target_width]
             player_resized = p_crop
        else:
             player_resized[0:target_height, p_start_x:p_start_x + new_pw] = player_resized_temp

        # 4. Animation
        gif_frames = []
        grid_x, grid_y = np.meshgrid(np.arange(target_width), np.arange(target_height))

        # Check alpha channel presence for blending
        if player_resized.shape[2] < 4:
            print(f"   ⚠️ {file_name} has NO alpha channel. Blending will be skipped.")

        for f in range(30):
            map_x = grid_x.astype(np.float32) + np.sin(grid_y / 40.0 + f / 5.0).astype(np.float32) * (target_width * 0.01)
            map_y = grid_y.astype(np.float32)
            distorted_bg = cv2.remap(logo_final, map_x, map_y, cv2.INTER_LINEAR)
            
            if player_resized.shape[2] < 4:
                canvas = player_resized.copy() 
            else:
                p_bgr = player_resized[:, :, 0:3]
                p_alpha = player_resized[:, :, 3] / 255.0
                p_alpha_3ch = cv2.merge([p_alpha, p_alpha, p_alpha])

                bg_part = (distorted_bg[:, :, 0:3] * (1.0 - p_alpha_3ch)).astype(np.uint8)
                fg_part = (p_bgr * p_alpha_3ch).astype(np.uint8)
                combined_rgb = cv2.add(bg_part, fg_part)
                canvas = cv2.merge([combined_rgb[:,:,0], combined_rgb[:,:,1], combined_rgb[:,:,2], player_resized[:, :, 3]])

            gif_frames.append(Image.fromarray(cv2.cvtColor(canvas, cv2.COLOR_BGRA2RGBA)))

        # 5. Save
        output_path = os.path.join(output_folder, f"{file_name}.gif")
        gif_frames[0].save(output_path, save_all=True, append_images=gif_frames[1:], duration=40, loop=0, disposal=2)
        
        # Verify file size
        f_size = os.path.getsize(output_path) / 1024
        print(f"   💾 Saved: {file_name}.gif | Size: {f_size:.1f} KB")

# Run

batch_process_to_gif(
    input_folder='/Users/vik/Downloads/testfolder/player_cartoons', 
    logo_path='/Users/vik/Downloads/testfolder/England.png', 
    output_folder='/Users/vik/Downloads/testfolder/player_gifs'
)


🖼️ Logo Loaded: /Users/vik/Downloads/testfolder/England.png | Shape: (2000, 1012, 4)
🔍 Found 29 files. Starting processing...
Processing: ollierobinson | Orig: 800x800 | Channels: 4
   📏 Width Comparison: Logo Canvas=1024px | Player=1024px
   📍 Placing player at x-coordinate: 0
   💾 Saved: ollierobinson.gif | Size: 2078.3 KB
Processing: lukewood | Orig: 480x480 | Channels: 4
   📏 Width Comparison: Logo Canvas=1024px | Player=1024px
   📍 Placing player at x-coordinate: 0
   💾 Saved: lukewood.gif | Size: 3203.9 KB
Processing: shoaibbashir | Orig: 800x800 | Channels: 4
   📏 Width Comparison: Logo Canvas=1024px | Player=1024px
   📍 Placing player at x-coordinate: 0
   💾 Saved: shoaibbashir.gif | Size: 3069.7 KB
Processing: joeroot | Orig: 800x800 | Channels: 4
   📏 Width Comparison: Logo Canvas=1024px | Player=1024px
   📍 Placing player at x-coordinate: 0
   💾 Saved: joeroot.gif | Size: 3137.8 KB
Processing: jofraarcher | Orig: 480x480 | Channels: 4
   📏 Width Comparison: Logo Canvas=1024p

In [10]:
import cv2
import numpy as np
import os
import glob

def batch_process_to_png(input_folder, logo_path, output_folder, target_height=1024, target_width=1024):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Load Logo with Alpha
    logo_img = cv2.imread(logo_path, cv2.IMREAD_UNCHANGED)
    if logo_img is None:
        print(f"❌ Error: Could not read logo at {logo_path}")
        return

    # 1. Prepare Logo Icon (Maintain Aspect Ratio & Larger Size)
    icon_h = 200  # Increased size
    l_h, l_w = logo_img.shape[:2]
    aspect_ratio = l_w / l_h
    icon_w = int(icon_h * aspect_ratio)
    logo_icon = cv2.resize(logo_img, (icon_w, icon_h), interpolation=cv2.INTER_LANCZOS4)

    player_paths = glob.glob(os.path.join(input_folder, "*.png"))
    print(f"🚀 Processing {len(player_paths)} players into static PNGs...")

    for p_path in player_paths:
        file_name = os.path.splitext(os.path.basename(p_path))[0]
        player_img = cv2.imread(p_path, cv2.IMREAD_UNCHANGED)
        if player_img is None: continue

        # 2. Player Resize & Centering
        p_h, p_w = player_img.shape[:2]
        scale = target_height / p_h
        new_pw = int(p_w * scale)
        player_resized_temp = cv2.resize(player_img, (new_pw, target_height), interpolation=cv2.INTER_LANCZOS4)
        
        # Create final transparent canvas
        # Note: Using 0,0,0,0 (transparent) as base. 
        # Change to 255,255,255,255 for solid white background.
        final_frame = np.zeros((target_height, target_width, 4), dtype=np.uint8)
        
        p_start_x = (target_width - new_pw) // 2
        
        # Place Player
        if new_pw > target_width:
             player_slice = player_resized_temp[0:target_height, (new_pw-target_width)//2 : (new_pw-target_width)//2 + target_width]
             final_frame = player_slice
        else:
             final_frame[0:target_height, p_start_x:p_start_x + new_pw] = player_resized_temp

        # 3. Overlay Logo Icon (Top Left Corner)
        # 20px padding from the top and left
        y_off, x_off = 20, 20
        
        # Split channels for alpha blending the icon
        icon_bgr = logo_icon[:, :, 0:3]
        icon_alpha = logo_icon[:, :, 3] / 255.0
        icon_alpha_3ch = cv2.merge([icon_alpha, icon_alpha, icon_alpha])

        # Define the ROI (Region of Interest) on the final frame
        roi = final_frame[y_off:y_off+icon_h, x_off:x_off+icon_w]
        
        # Blend: (Icon * IconAlpha) + (Background * (1 - IconAlpha))
        # This preserves the background player/transparency behind the icon
        bg_part = (roi[:, :, 0:3] * (1.0 - icon_alpha_3ch)).astype(np.uint8)
        fg_part = (icon_bgr * icon_alpha_3ch).astype(np.uint8)
        
        # Update BGR channels
        final_frame[y_off:y_off+icon_h, x_off:x_off+icon_w, 0:3] = cv2.add(bg_part, fg_part)
        # Update Alpha channel to ensure icon area isn't "cut out"
        final_frame[y_off:y_off+icon_h, x_off:x_off+icon_w, 3] = np.maximum(roi[:, :, 3], logo_icon[:, :, 3])

        # 4. Save as PNG
        output_path = os.path.join(output_folder, f"{file_name}.png")
        cv2.imwrite(output_path, final_frame)
        print(f"✅ Created: {file_name}.png")

# Run
batch_process_to_png(
    input_folder='/Users/vik/Downloads/testfolder/player_cartoons', 
    logo_path='/Users/vik/Downloads/testfolder/England.png', 
    output_folder='/Users/vik/Downloads/testfolder/player_gifs'
)

🚀 Processing 29 players into static PNGs...
✅ Created: ollierobinson.png
✅ Created: lukewood.png
✅ Created: shoaibbashir.png
✅ Created: joeroot.png
✅ Created: jofraarcher.png
✅ Created: chriswoakes.png
✅ Created: brydoncarse.png
✅ Created: markwood.png
✅ Created: adilrashid.png
✅ Created: rehanahmed.png
✅ Created: gusatkinson.png
✅ Created: samcurran.png
✅ Created: dawidmalan.png
✅ Created: philsalt.png
✅ Created: josbuttler.png
✅ Created: joshtongue.png
✅ Created: liamlivingstone.png
✅ Created: moeenali.png
✅ Created: benstokes.png
✅ Created: willjacks.png
✅ Created: chrisjordan.png
✅ Created: jamieoverton.png
✅ Created: jacobbethell.png
✅ Created: harrybrook.png
✅ Created: jonnybairstow.png
✅ Created: olliepope.png
✅ Created: liamdawson.png
✅ Created: benduckett.png
✅ Created: tombanton.png


In [8]:
import cv2
import numpy as np
import os
import glob
from PIL import Image

def batch_process_to_gif(input_folder, logo_path, output_folder, target_height=1024):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    logo_img = cv2.imread(logo_path, cv2.IMREAD_UNCHANGED)
    if logo_img is None:
        print(f"❌ Error: Could not read logo at {logo_path}")
        return

    player_paths = glob.glob(os.path.join(input_folder, "*.png"))
    print(f"🔍 Found {len(player_paths)} files. Starting consistent sizing...")

    for p_path in player_paths:
        player_img = cv2.imread(p_path, cv2.IMREAD_UNCHANGED)
        if player_img is None: continue

        file_name = os.path.splitext(os.path.basename(p_path))[0]
        
        # 1. Player Resize
        p_h, p_w = player_img.shape[:2]
        p_aspect = p_w / p_h
        canvas_h = target_height
        canvas_w = int(target_height * p_aspect)
        player_resized = cv2.resize(player_img, (canvas_w, canvas_h), interpolation=cv2.INTER_LANCZOS4)

        # 2. NEW LOGO LOGIC: Consistent Scale
        l_h, l_w = logo_img.shape[:2]
        # Match logo height to canvas height exactly
        logo_scale = canvas_h / l_h
        new_lw = int(l_w * logo_scale)
        logo_rescaled = cv2.resize(logo_img, (new_lw, canvas_h), interpolation=cv2.INTER_AREA)
        
        # 3. CREATE A MATCHING BACKGROUND SLICE
        # If logo is wider than player, crop the center. 
        # If player is wider than logo (unlikely with your 1024x2000), tile it or pad it.
        bg_canvas = np.zeros((canvas_h, canvas_w, 4), dtype=np.uint8)
        
        if new_lw >= canvas_w:
            # Crop center of logo to fit player width
            start_x = (new_lw - canvas_w) // 2
            logo_final = logo_rescaled[0:canvas_h, start_x:start_x + canvas_w]
        else:
            # Logo is too skinny? Center it on a black/transparent background
            start_x = (canvas_w - new_lw) // 2
            bg_canvas[0:canvas_h, start_x:start_x + new_lw] = logo_rescaled
            logo_final = bg_canvas

        # 4. Animation
        gif_frames = []
        grid_x, grid_y = np.meshgrid(np.arange(canvas_w), np.arange(canvas_h))

        for f in range(30):
            map_x = grid_x.astype(np.float32) + np.sin(grid_y / 40.0 + f / 5.0).astype(np.float32) * (canvas_w * 0.01)
            map_y = grid_y.astype(np.float32)
            distorted_bg = cv2.remap(logo_final, map_x, map_y, cv2.INTER_LINEAR)
            
            # Blend
            canvas = distorted_bg.copy()
            p_alpha = player_resized[:, :, 3] / 255.0
            
            for c in range(3):
                canvas[:, :, c] = (player_resized[:, :, c] * p_alpha + canvas[:, :, c] * (1 - p_alpha)).astype(np.uint8)
            canvas[:, :, 3] = np.maximum(canvas[:, :, 3], player_resized[:, :, 3])

            gif_frames.append(Image.fromarray(cv2.cvtColor(canvas, cv2.COLOR_BGRA2RGBA)))

        # 5. Save
        output_path = os.path.join(output_folder, f"{file_name}.gif")
        gif_frames[0].save(output_path, save_all=True, append_images=gif_frames[1:], duration=40, loop=0, disposal=2)
        print(f"✅ Finished: {file_name}")

# Run

batch_process_to_gif(
    input_folder='/Users/vik/Downloads/testfolder/player_cartoons', 
    logo_path='/Users/vik/Downloads/testfolder/England.png', 
    output_folder='/Users/vik/Downloads/testfolder/player_gifs'
)

🔍 Found 29 files. Starting consistent sizing...
✅ Finished: ollierobinson
✅ Finished: lukewood
✅ Finished: shoaibbashir
✅ Finished: joeroot
✅ Finished: jofraarcher


KeyboardInterrupt: 